# Lecture 2m: 2D Non-homogeneous Poisson Process on a Rectangle
**Simulations in Statistics (52001)**

---

## Overview

`Part2m(Lambda, mu, Sigma, I1, I2)` returns the points of a 2D non-homogeneous Poisson process restricted to the rectangle $I_1 \times I_2 \subset \mathbb{R}^2$. The rate function on $\mathbb{R}^2$ is proportional to the bivariate Normal density

$$\lambda(\vec{x}) \propto \exp\!\left(-\tfrac{1}{2}(\vec{x} - \vec{\mu})^\top \Sigma^{-1}(\vec{x} - \vec{\mu})\right)$$

with expected total count $\Lambda$ over $\mathbb{R}^2$.

**Construction:**

1. Draw $N \sim \operatorname{Poisson}(\Lambda)$.
2. If $N = 0$, return a $0 \times 2$ matrix.
3. Draw $\vec{X}_1, \ldots, \vec{X}_N \stackrel{\text{iid}}{\sim} \mathcal{N}_2(\vec{\mu}, \Sigma)$ via `MASS::mvrnorm`.
4. Restrict to $I_1 \times I_2$: keep only points with both coordinates inside. Use `drop = FALSE` to preserve matrix shape even if 0 or 1 row remain.

## R Implementation

In [4]:
suppressMessages(library(MASS))

Part2m <- function(Lambda, mu, Sigma, I1, I2) {
  poisson_total <- rpois(1, Lambda)
  if (poisson_total == 0) return(matrix(numeric(0), nrow = 0, ncol = 2))
  candidate_x <- mvrnorm(poisson_total, mu = mu, Sigma = Sigma)
  if (poisson_total == 1) candidate_x <- matrix(candidate_x, nrow = 1, ncol = 2)
  inside_rect <- candidate_x[, 1] >= I1[1] & candidate_x[, 1] <= I1[2] &
                 candidate_x[, 2] >= I2[1] & candidate_x[, 2] <= I2[2]
  return(candidate_x[inside_rect, , drop = FALSE])
}

In [5]:
set.seed(1)

total_intensity <- 23.7
mu_vec          <- c(4.08, 8.07)
Sigma_mat       <- matrix(c(
  1.000, 0.288,
  0.288, 1.000
), nrow = 2, ncol = 2, byrow = TRUE)
interval_I1     <- c(3.34, 5.71)
interval_I2     <- c(7.08, 9.21)
split_point_b   <- 4.5
n_replicates    <- 1e5

cat("Lambda    =", total_intensity, "\n")
cat("mu        =", mu_vec,          "\n")
cat("Sigma:\n");    print(Sigma_mat)
cat("I1        =", interval_I1,     "\n")
cat("I2        =", interval_I2,     "\n")
cat("b         =", split_point_b,   "\n")
cat("reps      =", n_replicates,    "\n")

Lambda    = 23.7 
mu        = 4.08 8.07 
Sigma:
      [,1]  [,2]
[1,] 1.000 0.288
[2,] 0.288 1.000
I1        = 3.34 5.71 
I2        = 7.08 9.21 
b         = 4.5 
reps      = 1e+05 


## Computing the Results

In [6]:
test.function <- function(X, b) {
  N1 <- sum(X[, 1] <= b)
  N2 <- sum(X[, 1] >  b)
  return(c(N1, N2))
}

temp <- replicate(
  n_replicates,
  test.function(
    Part2m(
      Lambda = total_intensity,
      mu     = mu_vec,
      Sigma  = Sigma_mat,
      I1     = interval_I1,
      I2     = interval_I2
    ),
    b = split_point_b
  )
)
N1 <- temp[1, ]
N2 <- temp[2, ]

freq_N1_eq_7 <- mean(N1 == 7)
freq_N2_eq_5 <- mean(N2 == 5)
correlation  <- cor(N1, N2)

cat("dim(temp)        =", dim(temp), "\n")
cat("a. mean(N1 == 7) =", freq_N1_eq_7, "\n")
cat("b. mean(N2 == 5) =", freq_N2_eq_5, "\n")
cat("c. cor(N1, N2)   =", correlation,  "\n")

dim(temp)        = 2 100000 
a. mean(N1 == 7) = 0.14822 
b. mean(N2 == 5) = 0.17419 
c. cor(N1, N2)   = 0.001413755 
